In [ ]:
# Getting relevant libraries
# VERIFY ALL INTENSIVE PHASES SUM TO ONE AND THAT THEY ARE NOT FORCED TO
import numpy as np 
import random
import matplotlib.pyplot as plt
import math
import pickle
import os
import pandas as pd
import random
plt.rcParams['figure.figsize'] = [10, 7]
from matplotlib.colors import LinearSegmentedColormap
#import mpl_scatter_density # adds projection='scatter_density'
from scipy.stats import gaussian_kde
from scipy import optimize
from molmass import Formula
import csv
import re
import copy
import gc
import time
import molmass as ms
from tqdm import tqdm
#from Emulator1_0_2_Aug21_2025 import Emulator102GPU_Cr, Emulator102CPU_Cr, Emulator102GPU_NoCr, Emulator102CPU_NoCr
#import Emulator1_0_2_Sept12_2025 as Em
from BackEnds.EmulatorLibrary import *
from BackEnds.nnMELTS import rebuild_MELTS_model, NN_MELTS


MELTSModel= '102'
CalcType = 'Batch'
date = 'Nov9'

DictFilePaths=[f"Models/MELTS{MELTSModel}{CalcType}{['NoCr', 'Cr'][i]}_Final_{date}.pt" for i in range(2)]

"""GPUFullMELTS_Cr = DualSaturationChemistry().cuda()
GPUFullMELTS_Cr.load_state_dict(torch.load(DictFilePaths[1])) 

GPUFullMELTS_NoCr = DualSaturationChemistry()
GPUFullMELTS_NoCr.load_state_dict(torch.load(DictFilePaths[0]))

CPUFullMELTS_Cr = DualSaturationChemistry().cuda()
CPUFullMELTS_Cr.load_state_dict(torch.load(DictFilePaths[1]), strict = False)

CPUFullMELTS_NoCr = DualSaturationChemistry()
CPUFullMELTS_NoCr.load_state_dict(torch.load(DictFilePaths[0]), strict = False)

EmulatorGPU_Cr = NN_MELTS(GPUFullMELTS_Cr, cuda = True)
EmulatorGPU_NoCr = NN_MELTS(GPUFullMELTS_NoCr, cuda = True)
EmulatorCPU_Cr = NN_MELTS(CPUFullMELTS_Cr, cuda = False)
EmulatorCPU_NoCr = NN_MELTS(CPUFullMELTS_NoCr, cuda = False)"""

In [ ]:
### Loading Validation Data:
modelNo = 102
date = 'Nov9'
calctype = 'Batch'


Validfilename = f'D:/Workspace/{modelNo}Datasets/MELTS{modelNo}_Validset{date}{calctype}Cooling'

PTfO2min = torch.tensor([1,700,-5], device = 'cpu', dtype = torch.float)
PTfO2max = torch.tensor([10000,2000,5], device = 'cpu', dtype = torch.float)
min_tensor = torch.zeros(len(Elkeys)+3, device = 'cpu', dtype = torch.float)
min_tensor[:3] = PTfO2min
range_tensor = torch.ones(len(Elkeys)+3, device = 'cpu', dtype = torch.float)
range_tensor[:3] = PTfO2max - PTfO2min


feature_path = Validfilename+'features.npy'
binary_path  = Validfilename+'binary_labels.npy'
label_path = Validfilename+'labels.npy'
mass_path = Validfilename+'mass_labels.npy'
mole_path = Validfilename+'molar_labels.npy'
featureMap = np.load(feature_path, mmap_mode='r')
binaryMap = np.load(binary_path, mmap_mode='r')
labelMap = np.load(label_path, mmap_mode='r')
massMap = np.load(mass_path, mmap_mode='r')
moleMap = np.load(mole_path, mmap_mode='r')


normf = Normalizer(min_tensor=min_tensor, range_tensor=range_tensor)

validation_features = normf.norm(torch.tensor(featureMap, device = 'cpu', dtype = torch.float)).detach().numpy()
del featureMap
gc.collect()

validation_binaries = torch.tensor(binaryMap, device = 'cpu', dtype = torch.float).detach().numpy()
del binaryMap
gc.collect()

validation_labels = torch.tensor(labelMap, device = 'cpu', dtype = torch.float).detach().numpy()
validation_labels_trans = torch.tensor(labelMap, device = 'cpu', dtype = torch.float).detach().numpy() @ PxSpTransform[np.ix_(compositional_component_subset, compositional_component_subset)]
del labelMap
gc.collect()

validation_masses = torch.tensor(massMap, device = 'cpu', dtype=torch.float).detach().numpy()
del massMap
gc.collect()

validation_moles = torch.tensor(moleMap, device = 'cpu', dtype=torch.float).detach().numpy()
del moleMap
gc.collect()

#  Remove small fraction of assemblages that do not reconstruct bulk
bulk_wt_ox = (validation_features[:,3:] @ np.linalg.inv(oxToEl[:-1])) @ MM[:-1,:-1]
bulk_wt_ox = 100*bulk_wt_ox/np.sum(bulk_wt_ox, axis = 1).reshape(-1,1)

GT_comps = np.zeros((validation_features.shape[0],label_indices['melts-liquid'][-1]+1))

for phase in np.array(list(label_indices.keys())):
    if phase in compositionally_variable_phases:
        GT_comps[:,label_indices[phase]] = (validation_moles[:, mass_phasedict[phase]]).reshape(-1,1) * validation_labels_trans[:,label_indices_comp[phase]]
    else:
        GT_comps[:,label_indices[phase]] = (validation_moles[:, mass_phasedict[phase]]).reshape(-1,1)

GTReconBulk_oxides = (((GT_comps @ compToOx) @ oxToEl) @ np.linalg.inv(oxToEl[:-1])) @ MM[:-1,:-1]
GTReconBulk_oxides *= 100/np.sum(GTReconBulk_oxides,axis=1, keepdims=True)

mismatches = np.unique(np.where(np.round(bulk_wt_ox,2) != np.round(GTReconBulk_oxides,2))[0])
mismatch_mask = np.zeros(bulk_wt_ox.shape[0]).astype(bool)
mismatch_mask[mismatches] = True
print(mismatches.shape)
print(bulk_wt_ox.shape)

#Remove assemblages with out-of-bounds pyroxenes (~1%)
OOB = ((validation_labels_trans > 1).astype(float) + (validation_labels_trans < 0).astype(float)).astype(bool)
badMap = np.unique(np.where(OOB)[0])
#goodMap = np.arange(validation_labels_trans.shape[0])
goodMap = np.ones(validation_features.shape[0]).astype(bool)
#goodMap = goodMap[~np.isin(goodMap, badMap)] # Exclude OOB IDs
goodMap[badMap] = False
goodMap[mismatch_mask] = False

print(f"Validation Features: {validation_features.shape}, Binaries {validation_binaries.shape}, labels: {validation_labels.shape}")
validation_features, validation_binaries, validation_labels, validation_labels_trans, validation_masses, validation_moles = validation_features[goodMap], validation_binaries[goodMap], validation_labels[goodMap], validation_labels_trans[goodMap], validation_masses[goodMap], validation_moles[goodMap]
print(f"Validation Features: {validation_features.shape}, Binaries {validation_binaries.shape}, labels: {validation_labels.shape}")


# IncludeCr free assemblages with rare phases
Cr_in = validation_features[:,-1] != 0 + np.any(
    validation_binaries[:,torch.tensor([mass_phasedict[phase] for phase in ['nepheline', 'leucite', 'analcime', 'alloy-solid', 'muscovite', 'k-feldspar']])] > 0.5, 
    axis = -1).astype(bool)
rhm_idx = np.where(validation_binaries[:,mass_phasedict['rhm-oxide']] > 0.5)[0]
Cr_in[rhm_idx[np.random.choice(np.arange(len(rhm_idx)), size = len(rhm_idx)//3)]] = True # Add back 1/3rd of rhm-oxides. 

Cr_out = (validation_features[:,-1] == 0).astype(bool)
print(f"Chrome in Validation: {Cr_in.sum()}, Chrome Absent in Validation: {Cr_out.sum()}")


#assert mismatches.shape[0] == 0

In [ ]:
import importlib

#from Emulator1_0_2_Sept12_2025 import Emulator102CPU_Cr

# Make changes to my_module.py externally

#importlib.reload(Em)
import BackEnds.nnMELTS as NN
importlib.reload(NN)

DictFilePaths=[f"Models/MELTS{MELTSModel}{CalcType}{['NoCr', 'Cr'][i]}_Final_{date}.pt" for i in range(2)]


# Plotting chrome-bearing and chrome free emulator results *With Masses*
for i, CrBools in enumerate([Cr_in, Cr_out]):
    CrText = ["Cr","NoCr"][i]
    directory = f'Plots/MELTS{modelNo}_Validation{date}{calctype}_{CrText}_ChemMole'
    #if i ==0: 
        #continue
    Emulator102CPU = NN.NN_MELTS(rebuild_MELTS_model(DictFilePaths[i]), cuda = False)
    Emulator102GPU = NN.NN_MELTS(rebuild_MELTS_model(DictFilePaths[i]), cuda = True)

    #Emulator102GPU = [Em.GPUFullMELTS_Cr, Em.GPUFullMELTS_NoCr][i]
    
    #Emulator102CPU.load_state_dict(torch.load(f'Models/MELTS{modelNo}{calctype}{CrText}_FullL2_{date}.pt'))
    #Emulator102GPU.load_state_dict(torch.load(f'Models/MELTS{modelNo}{calctype}{CrText}_FullL2_{date}.pt'))

    if not os.path.exists(directory):
        os.makedirs(directory)

    to_plot_no = 2**16
    inds = np.where(CrBools & (validation_masses[:,-1] > 1))[0]
    if len(inds) > to_plot_no:
        #subset = np.random.choice(np.arange(0, len(validation_binaries)), size=100000, replace=False)
        subset = np.random.choice(inds, size=to_plot_no, replace=False)
        
    else: 
        subset = inds

    with torch.no_grad():
        begin_time = time.time()
        liklihoods, transcomponent_hat, phaseMoles, reconBulk, componentMoles, phaseProportions = Emulator102GPU.model.forward(
            torch.tensor(validation_features[subset], device = 'cuda'), detailed=True)#, WtPercent=False, Normalize = False)
        NN_time = time.time()
        (compTens, massTens), componentMoles2, wtDelComponentMoles = Emulator102GPU.polish_masses(phaseMoles, reconBulk, componentMoles, phaseProportions, 
                                                          features=torch.tensor(validation_features[subset], device = 'cuda'), 
                                                          optimize_masses=False, output_componentMoles=True, protect_opx=False)
        residuals = validation_features[subset,3:] - reconBulk.detach().cpu().numpy()
        Linear_Algebra_time = time.time()
        print(f"NN Time: {NN_time-begin_time} sec")
        print(f"Linear_algebra_time: {Linear_Algebra_time-NN_time} sec")
        print(f"Total Time: {Linear_Algebra_time-begin_time}")
    binary_hat = (liklihoods > 0.5).float().detach()
    liklihoods = liklihoods.detach().cpu().numpy()
    
    transcomponent_hat = transcomponent_hat.detach().cpu().numpy()
    phaseMoles = phaseMoles.detach().cpu().numpy()
    component_hat = transcomponent_hat @ np.linalg.inv(PxSpTransform[np.ix_(compositional_component_subset,compositional_component_subset)])
    binary_hat = binary_hat.cpu().numpy()

    correct_phases = np.all((validation_binaries[subset] > 0.5).astype(float) == binary_hat, axis = 1)   
    print(np.shape(correct_phases))
    datalen = len(component_hat)
    j = 0
    gc.collect()
    
    

    for i, (phase, indices) in enumerate(label_indices.items()):
        realPos = (validation_binaries[subset,i] == 1)
        predPos = binary_hat[:,i]
        plotable = np.logical_or(realPos, predPos)
        FP = np.sum(realPos[plotable]==0)/len(realPos[plotable])
        FN = np.sum(predPos[plotable]==0)/len(predPos[plotable])
        prop_correct_plotable = 100*np.sum(plotable*correct_phases)/np.sum(plotable)
        statable = (realPos*predPos).astype(bool)
        binary_condition = (statable*correct_phases).astype(bool) # condition for correct binary output and phase present

        plt.hist(liklihoods[realPos.astype(bool),i], bins=30, alpha=0.5, color = 'blue', label=f'{phase} Present', density=True, log = True)
        plt.hist(liklihoods[~(realPos.astype(bool)),i], bins=30, alpha=0.5, color = 'red', label=f'{phase} Absent', density=True, log = True)
        #plt.axvline(x=3, color='r', linestyle='dashed', linewidth=1)
        plt.legend()
        plt.xlabel("Probability")
        plt.ylabel("Normalized Frequency (Log Scale)")
        plt.title(f"NN {phase} Saturation Probabilities, {CrText} Model\nPresent in {round(100*realPos.sum()/datalen,2)}% of Dataset")
        plt.tight_layout()
        plt.savefig(directory+f"/{phase}_Saturation_Probability_Histogram")
        plt.show()
        
        # Plot Masses! First w/o gradient descent, then with
        
        name = phase + 'System Mass (consistent)'
        plt.title(f'{name} 1:1, {CrText} Model')
        plt.scatter(validation_masses[subset[plotable*~correct_phases],i],massTens[plotable*~correct_phases,i], color = 'red', s = 0.1*(datalen/np.sum(realPos)), alpha = 0.2, label =f'Assemblage Miss ({100 - prop_correct_plotable}%)')
        plt.scatter(validation_masses[subset[plotable*correct_phases],i],massTens[plotable*correct_phases,i], color = 'blue', s = 0.1*(datalen/np.sum(realPos)), alpha = 0.2, label =f'Complete Assemblage Recovered ({prop_correct_plotable}%)')
        plt.xlabel(f'GT Wt% {phase}, FN = {round(100*FN,2)}%')
        plt.ylabel(f'Predicted Wt% {phase}, FP = {round(100*FP,2)}%')
        xlim = plt.xlim()
        #ylim = plt.ylim()
        plt.plot([xlim[0],xlim[1]], [xlim[0],xlim[1]], linestyle = '--', color = 'black')
        plt.legend()
        plt.tight_layout()
        plt.savefig(directory + '/ '[0] + name + '.jpg', dpi = 256)
        plt.show()
        
        name = phase + ' Moles, Unconstrained NN Output'
        plt.title(f'{name} 1:1, {CrText} Model')
        plt.scatter(validation_moles[subset[plotable*~correct_phases],i],phaseMoles[plotable*~correct_phases,i], color = 'red', s = 0.3*(datalen/np.sum(realPos)), alpha = 0.2, label =f'Assemblage Miss ({100 - prop_correct_plotable}%)')
        plt.scatter(validation_moles[subset[plotable*correct_phases],i],phaseMoles[plotable*correct_phases,i], color = 'blue', s = 0.3*(datalen/np.sum(realPos)), alpha = 0.2, label =f'Complete Assemblage Recovered ({prop_correct_plotable}%)')
        plt.xlabel(f'GT Moles {phase}, FN = {round(100*FN,2)}%')
        plt.ylabel(f'Predicted Moles {phase}, FP = {round(100*FP,2)}%')
        xlim = plt.xlim()
        #ylim = plt.ylim()
        plt.plot([xlim[0],xlim[1]], [xlim[0],xlim[1]], linestyle = '--', color = 'black')
        plt.legend()
        plt.tight_layout()
        plt.savefig(directory + '/ '[0] + name + '.jpg', dpi = 256)
        plt.show()
        
        if phase in list(label_indices_comp.keys()):
            # First, Molar quantities. Then mass after speciating iron in liquid
            oxides_hat = (transcomponent_hat[:,label_indices_comp[phase]] @ compToOx[indices])
            oxides_GT = (validation_labels_trans[np.ix_(subset,label_indices_comp[phase])] @ compToOx[indices])
            if phase == 'melts-liquid':
                oxides_GT = Emulator102GPU.Iron_Speciator(torch.tensor(oxides_GT, device = 'cuda', dtype = torch.float32), torch.tensor(validation_features[subset], device = 'cuda', dtype = torch.float32))
                oxides_GT = oxides_GT.detach().cpu().numpy()
                oxides_hat = Emulator102GPU.Iron_Speciator(torch.tensor(oxides_hat, device = 'cuda', dtype = torch.float32), torch.tensor(validation_features[subset], device = 'cuda', dtype = torch.float32))
                oxides_hat = oxides_hat.detach().cpu().numpy()
                
            # Convert to mass
            oxides_hat = oxides_hat @ MM
            oxides_GT = oxides_GT @ MM
            oxides_GT = oxides_GT * (100/np.sum(oxides_GT,axis=1)).reshape(-1,1)
            oxides_hat = oxides_hat * (100/np.sum(oxides_hat,axis=1)).reshape(-1,1)
            
            for oxInd in active_ox_dict[phase]:
                """Plot Oxides as predicted by NN"""
                name = phase + ' Unconstrained NN ' + Oxides[oxInd]
                plt.title(f'{name} 1:1 Plot, {CrText} Model')
                plt.scatter(oxides_GT[plotable*~correct_phases,oxInd],oxides_hat[plotable*~correct_phases,oxInd], color = 'red', s = 0.3*(datalen/np.sum(realPos)), alpha = 0.2, label =f'Assemblage Miss ({100 - prop_correct_plotable}%)')
                plt.scatter(oxides_GT[plotable*correct_phases,oxInd],oxides_hat[plotable*correct_phases,oxInd], color = 'blue', s = 0.3*(datalen/np.sum(realPos)), alpha = 0.2, label =f'Complete Assemblage Recovered ({prop_correct_plotable}%)')
                plt.xlabel(f'True {Oxides[oxInd]} wt%, FN = {round(100*FN,2)}%')
                plt.ylabel(f'Predicted {Oxides[oxInd]}  wt%,  FP = {round(100*FP,2)}%')
                xlim = plt.xlim()
                #ylim = plt.ylim()
                plt.plot([xlim[0],xlim[1]], [xlim[0],xlim[1]], linestyle = '--', color = 'black')
                plt.legend()
                plt.tight_layout()
                plt.savefig(directory + '/ '[0] + name + '.jpg', dpi = 256)
                plt.show()
                
                # Now plot oxides with linear algebra to get masses and consistent compositions
                
                # Grab data
                xdata = oxides_GT[plotable, oxInd]
                ydata = compTens[plotable, comp_phasedict[phase], oxInd].detach().cpu().numpy()

                # Compute min/max with padding
                xmin = max(0, np.nanmin(xdata))
                xmax = min(100, np.nanmax(xdata))
                ymin = max(0, np.nanmin(ydata).item())
                ymax = min(100, np.nanmax(ydata).item())

                # Add padding (5% of the range)
                xpad = 0.05 * (xmax - xmin) if xmax > xmin else 1
                ypad = 0.05 * (ymax - ymin) if ymax > ymin else 1

                xmin, xmax = xmin - xpad, xmax + xpad
                ymin, ymax = ymin - ypad, ymax + ypad

                # Count off-plot points
                offplot = (
                    ((xdata < xmin) | (xdata > xmax)) |
                    ((ydata < ymin) | (ydata > ymax))
                ).sum().item()

                # Apply to plot

                name = phase + ' Consistent ' + Oxides[oxInd]
                plt.title(f'{name} 1:1 Plot, {CrText} Model')
                plt.scatter(oxides_GT[plotable*~correct_phases,oxInd],compTens[plotable*~correct_phases,comp_phasedict[phase],oxInd], color = 'red', s = 0.3*(datalen/np.sum(realPos)), alpha = 0.2, label =f'Assemblage Miss ({100 - prop_correct_plotable}%)')
                plt.scatter(oxides_GT[plotable*correct_phases,oxInd],compTens[plotable*correct_phases,comp_phasedict[phase],oxInd], color = 'blue', s = 0.3*(datalen/np.sum(realPos)), alpha = 0.2, label =f'Complete Assemblage Recovered ({prop_correct_plotable}%)')
                plt.xlabel(f'True {Oxides[oxInd]} wt%, FN = {round(100*FN,2)}%')
                plt.xlim(xmin, xmax)
                plt.ylim(ymin, ymax)
                plt.ylabel(f'Predicted {Oxides[oxInd]} wt%, FP = {round(100*FP,2)}%, Off-plot = {100*offplot/np.sum(plotable)}%')
                
                #plt.ylim(-10,110)
                plt.plot([xlim[0],xlim[1]], [xlim[0],xlim[1]], linestyle = '--', color = 'black')
                plt.legend()
                plt.tight_layout()
                plt.savefig(directory + '/ '[0] + name + '.jpg', dpi = 256)
                plt.show()
                
            for k, ind in enumerate(label_indices_comp[phase]):
                """Plot Native Components"""
                name = phase + ' ' + label_names[label_indices[phase][k]]
                plt.title(f'{name} 1:1, Plot{CrText} Model')
                plt.scatter(validation_labels[subset[plotable*correct_phases],ind],component_hat[plotable*correct_phases,ind], color = 'blue', s = 0.3*(datalen/np.sum(realPos)), alpha = 0.2, label =f'Complete Assemblage Recovered ({prop_correct_plotable}%)')
                plt.xlabel(f'True molar {label_names[label_indices[phase][k]]}, FN = {round(100*FN,2)}%')
                plt.ylabel(f'Predicted molar {label_names[label_indices[phase][k]]}, FP = {round(100*FP,2)}%')
                xlim = plt.xlim()
                #ylim = plt.ylim()
                plt.plot([xlim[0],xlim[1]], [xlim[0],xlim[1]], linestyle = '--', color = 'black')
                plt.legend()
                plt.tight_layout()
                plt.savefig(directory + '/ '[0] + name + '.jpg', dpi = 256)
                plt.show()
            
                if phase in ['orthopyroxene', 'clinopyroxene', 'spinel']:
                    """Plot Transformed Components"""
                    name = phase + f' g{k-1} component' 
                    plt.title(f'{name} 1:1 Plot, {CrText} Model')
                    plt.scatter(validation_labels_trans[subset[plotable*~correct_phases],ind],transcomponent_hat[plotable*~correct_phases,ind], color = 'red', s = 0.3*(datalen/np.sum(realPos)), alpha = 0.2, label =f'Assemblage Miss ({100 - prop_correct_plotable}%)')
                    plt.scatter(validation_labels_trans[subset[plotable*correct_phases],ind],transcomponent_hat[plotable*correct_phases,ind], color = 'blue', s = 0.3*(datalen/np.sum(realPos)), alpha = 0.2, label =f'Complete Assemblage Recovered ({prop_correct_plotable}%)')
                    plt.xlabel(f'True molar g{k-1}, FN = {round(100*FN,2)}%')
                    plt.ylabel(f'Predicted molar g{k-1}, FP = {round(100*FP,2)}%')
                    xlim = plt.xlim()
                    #ylim = plt.ylim()
                    plt.plot([xlim[0],xlim[1]], [xlim[0],xlim[1]], linestyle = '--', color = 'black')
                    plt.legend()
                    plt.tight_layout()
                    plt.savefig(directory + '/ '[0] + name + '.jpg', dpi = 256)
                    plt.show()

In [ ]:
for i, NNtens in enumerate([liklihoods, transcomponent_hat, phaseMoles, reconBulk, componentMoles, phaseProportions]):
    varname = ['liklihoods', 'transcomponent_hat','phaseMoles', 'reconBulk', 'componentMoles', 'phaseProportions'][i]
    print(f"Shape of {varname}: {NNtens.size()}")
    boolans = torch.isnan(NNtens)
    print(f"NaNs in {varname}: {boolans.sum()}")
    idxAns = torch.where(boolans)
    print(f"NaN loc for {varname}: {idxAns}")
    boolans = torch.isinf(NNtens)
    print(f"inf in {varname}: {boolans.sum()}")
    idxAns = torch.where(boolans)
    print(f"inf loc for {varname}: {idxAns}")


In [ ]:
transcomponent_hat.sum(dim=0)

In [ ]:
susIDX = torch.where(torch.isnan(componentMoles))
susRows = torch.unique(susIDX[0])
componentMoles[susRows]
print(phaseProportions[susIDX])
plt.hist(phaseProportions[susRows].sum(-1).detach().cpu().numpy())
plt.hist(phaseProportions[~susRows].sum(-1).detach().cpu().numpy())

print(reconBulk[susRows])
print(liklihoods[susRows])


In [ ]:

### INCORPERATE MIDDLE LAYER
    """ 
    Forward pass for both training and inference.

    Args:
        x (Tensor): Input system representation [batch_size, input_dim]
        binaries (Tensor or None): If provided, used as ground-truth saturation labels.
                                    If None, saturation predictions are used.

    Returns:
        Tuple of (saturation logits or likelihoods, masked chemistry predictions)
    """

self = NN.NN_MELTS(rebuild_MELTS_model(DictFilePaths[0]), cuda = True)
x = validation_features[subset]

    # Encode features
    latent = self.encoder(x)

    # Build mask to exclude components that require elements not in inputs
    inf_mask = ((x[:,3:] == 0).to(torch.float32) @ self.boolTransCompToOx[self.compositionally_variable_subset].T.to(torch.float32)) != 0 #be,ec->bc 


    # Phase saturation logits (not yet sigmoid)
    """sat_outputs = [head(latent) for head in self.sat_heads]
    logits = torch.cat(sat_outputs, dim=1)"""
    logits = self.sat_head(latent)

    """likelihoods = torch.sigmoid(logits) # Testing superliquidus no-compute 10/10/25
    binary_pred = (likelihoods > 0.5).float()

    if binaries is None:
        # Inference mode — use predicted binaries
        binary_inp = binary_pred

        
    else:
        # Training mode — use provided ground truth binaries
        binary_inp = binaries

    # Construct masking matrix for chemistry predictions
    zero_mask = binary_inp[:, self.comp_binaries] @ self.comp_mappings  # [batch, n_components]

    if binaries is None: #Inference
        chem_outputs = [head(latent, inf_mask = inf_mask[:,(self.comp_mappings[i]).to(torch.bool)]) for i, head in enumerate(self.chem_heads)]
        chem_out = torch.cat(chem_outputs, dim=1) 
        if not NN_only:
            begin_refit = time.time()
            chem_out = self.polish_negative_px(chem_out)
            begin_spinel = time.time()
            chem_out = self.polish_negative_sp(chem_out)
            print(f"To solve 0CaO Px: {round((begin_spinel-begin_refit)*1E6)} microsec; To solve 0FeO/Al2O3 Sp: {round((time.time()-begin_spinel)*1E6)} microsec ")

        
    else: #Training
        chem_outputs = [head(latent, train_inf_mask = inf_mask[:,(self.comp_mappings[i]).to(torch.bool)]) for i, head in enumerate(self.chem_heads)]
        chem_out = torch.cat(chem_outputs, dim=1) 
    
    phaseMass, reconBulk, componentMoles, phaseProportions = self.forward_phase_moles(latent, binary_mask=binary_pred.detach(), intensiveComponents=chem_out, details_out=True)

    if binaries is None:
        if detailed:
            return likelihoods, chem_out*zero_mask, phaseMass, reconBulk, componentMoles, phaseProportions
        else:
            return likelihoods, chem_out*zero_mask, phaseMass, reconBulk # Inference
    else:
        return logits, chem_out*zero_mask, zero_mask, phaseMass, reconBulk # Training, return zero mask for loss masking of intensive chemistries"""

    likelihoods = torch.sigmoid(logits)
    binary_pred = (likelihoods > 0.5).float()

    # Identify superliquidus rows: only the last head > 0.5
    superliquidus = (binary_pred[:, :-1].sum(dim=1) == 0) & (binary_pred[:, -1] == 1)
    non_super = ~superliquidus

    if binaries is None:
        binary_inp = binary_pred
    else:
        binary_inp = binaries
    features = x  # alias for clarity

    if self.middleBrain is not None: # If there is a middle encoder, use it. Otherwise prepare first encodings and phase saturation
        CoreOutput = self.middleBrain(torch.cat([latent, binary_inp, features], dim=1))
    else: 
        CoreOutput = torch.cat([latent, binary_inp, features], dim=1)

    zero_mask = binary_inp[:, self.comp_binaries] @ self.comp_mappings  # [batch, n_components]

    if binaries is None:  # Inference
        chem_outputs = [
            head(CoreOutput, inf_mask=inf_mask[:, (self.comp_mappings[i]).to(torch.bool)])
            for i, head in enumerate(self.chem_heads)
        ]
        chem_out = torch.cat(chem_outputs, dim=1)

        # Zero out superliquidus rows
        chem_out[superliquidus] = 0.0

        # Overwrite liquid component columns with feature composition
        liq_idx = torch.tensor(label_indices_comp['melts-liquid'], device=chem_out.device)
        chem_out[superliquidus][:, liq_idx] = features[superliquidus, 3:]


        if not NN_only and non_super.any():
            chem_out[non_super] = self.polish_negative_px(chem_out[non_super])
            chem_out[non_super] = self.polish_negative_sp(chem_out[non_super])

    else:  # Training
        chem_outputs = [
            head(CoreOutput, train_inf_mask=inf_mask[:, (self.comp_mappings[i]).to(torch.bool)])
            for i, head in enumerate(self.chem_heads)
        ]
        chem_out = torch.cat(chem_outputs, dim=1)

        # Overwrite liquid component columns with feature composition
        liq_idx = torch.tensor(label_indices_comp['melts-liquid'], device=chem_out.device)
        chem_out[superliquidus][:, liq_idx] = features[superliquidus, 3:]


    # Compute phase properties
    phaseMass, reconBulk, componentMoles, phaseProportions = self.forward_phase_moles(
        CoreOutput, binary_mask=binary_pred.detach(), intensiveComponents=chem_out, details_out=True
    )

    # Assign direct values for superliquidus rows
    liq_idx_phase = torch.tensor(label_indices['melts-liquid'], device=chem_out.device)

    reconBulk[superliquidus] = features[superliquidus, 3:]
    componentMoles[superliquidus][:, liq_idx_phase] = features[superliquidus, 3:]
    phaseProportions[superliquidus][:, liq_idx_phase] = features[superliquidus, 3:]
    reconBulk[superliquidus] = features[superliquidus, 3:]
    phaseMass[superliquidus, -1] = 1.0


    if binaries is None:
        if detailed:
            return likelihoods, chem_out*zero_mask, phaseMass, reconBulk, componentMoles, phaseProportions # Inference with residual fitting
        else:
            return likelihoods, chem_out*zero_mask, phaseMass, reconBulk # Inference
    else:
        return logits, chem_out*zero_mask, zero_mask, phaseMass, reconBulk # Training, return zero mask for loss masking of intensive chemistries"""

